# Extended Kalman Filter (EKF) Results Analysis

This notebook loads the exported state and covariance matrices from the EKF simulation and compares them against the "Truth" receiver trajectory. It generates standard time-series plots to evaluate the filter's estimation errors and 3-sigma uncertainty bounds.

In [ ]:
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

In [ ]:
# Define paths based on the project structure
project_root = Path.cwd().parent
data_path = project_root / 'data'

# Paths to the specific run results
results_path = data_path / 'processed' / 'filter_results' / 'sample2'

# Paths to the truth data
rcver_path = data_path / 'raw' / 'receiver_orbits' / 'eor' / 'sample2.txt'
clock_path = data_path / 'interim' / 'clock_sim' / 'true_clock_bias_drift.npy'

In [ ]:
print("Loading EKF results...")
t_log_array = np.load(results_path / f"tlog.npy")
estim_states = np.load(results_path / f"states.npy")
P_matrices = np.load(results_path / f"covariances.npy")

print("Loading Truth trajectory...")
rcver_data = np.loadtxt(rcver_path, skiprows=1, usecols=(13, 14, 15, 16, 17, 18))
true_pos = 1e3 * rcver_data[:, 0:3]
true_vel = 1e3 * rcver_data[:, 3:6]

# Load Truth Clock States
clock_bias_drift = np.load(clock_path)
true_bias = clock_bias_drift[0,:]
true_drift = clock_bias_drift[1,:]

# Convert time array to hours for plotting
t_hours = t_log_array / 3600.0

print("Data loaded successfully.")

In [ ]:
# 1. 3D Position and Velocity Estimation Errors (Euclidean Norm)
pos_err = np.linalg.norm(estim_states[:, 0:3] - true_pos, axis=1)
vel_err = np.linalg.norm(estim_states[:, 3:6] - true_vel, axis=1)

# 2. Clock Estimation Errors
bias_err = estim_states[:, 6] - true_bias
drift_err = estim_states[:, 7] - true_drift

# 3. 3-Sigma Uncertainty Bounds (from the trace of the covariance blocks)
pos_std = np.sqrt(P_matrices[:, 0, 0] + P_matrices[:, 1, 1] + P_matrices[:, 2, 2])
vel_std = np.sqrt(P_matrices[:, 3, 3] + P_matrices[:, 4, 4] + P_matrices[:, 5, 5])
bias_std = np.sqrt(P_matrices[:, 6, 6])
drift_std = np.sqrt(P_matrices[:, 7, 7])

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 10), sharex=True)

# --- Position Error ---
ax1.semilogy(t_hours, pos_err, lw=1.5, color='tab:blue', label=r'$|\hat{\mathbf{r}} - \mathbf{r}_{true}|$')
ax1.semilogy(t_hours, 3 * pos_std, lw=1.5, ls='-.', color='tab:blue', label=r'$3\sigma$ Bound')
ax1.set_ylabel('3D Position Error [m]')
ax1.set_title(f'EKF Kinematic Estimation Errors')
ax1.grid(True, which='both', linestyle='--', alpha=0.6)
ax1.legend(loc='upper right')

# --- Velocity Error ---
ax2.semilogy(t_hours, vel_err, lw=1.5, color='tab:orange', label=r'$|\hat{\mathbf{v}} - \mathbf{v}_{true}|$')
ax2.semilogy(t_hours, 3 * vel_std, lw=1.5, ls='-.', color='tab:orange', label=r'$3\sigma$ Bound')
ax2.set_ylabel('3D Velocity Error [m/s]')
ax2.set_xlabel('Time [hours]')
ax2.grid(True, which='both', linestyle='--', alpha=0.6)
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 10), sharex=True)

# --- Clock Bias Error ---
ax1.plot(t_hours, bias_err, lw=1.5, color='tab:purple', label='Bias Error')
ax1.fill_between(t_hours, 3 * bias_std, -3 * bias_std, color='tab:purple', alpha=0.2, label=r'$\pm 3\sigma$ Bound')
ax1.set_ylabel('Clock Bias Error [m]')
ax1.set_title('EKF Receiver Clock Estimation Errors')
ax1.set_ylim(-10,10)
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend(loc='upper right')

# --- Clock Drift Error ---
ax2.plot(t_hours, drift_err, lw=1.5, color='tab:brown', label='Drift Error')
ax2.fill_between(t_hours, 3 * drift_std, -3 * drift_std, color='tab:brown', alpha=0.2, label=r'$\pm 3\sigma$ Bound')
ax2.set_ylabel('Clock Drift Error [m/s]')
ax2.set_xlabel('Time [hours]')
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
import csv
from scipy.stats import chi2

# Load Mahalanobis Data
mah_file = results_path / f'mahalanobis.csv'
mah_time, mah_dof, mah_sq = [], [], []

try:
    with open(mah_file, mode='r') as file:
        reader = csv.reader(file)
        next(reader)  # Skip header
        for row in reader:
            mah_time.append(float(row[0]) / 3600.0)
            mah_dof.append(float(row[1]))
            mah_sq.append(float(row[2]))
            
    # Calculate 95% dynamic threshold based on degrees of freedom (measurements)
    thresholds_95 = [chi2.ppf(0.95, df=m) for m in mah_dof]

    # Plot
    plt.figure(figsize=(10, 5))
    plt.semilogy(mah_time, mah_sq, '.', color='tab:blue', alpha=0.6, label=r'Mahalanobis $D^2$')
    plt.semilogy(mah_time, thresholds_95, '-', color='tab:red', drawstyle='steps-mid', label=r'95% $\chi^2$ Threshold')
    
    plt.xlabel('Time [hours]')
    plt.ylabel(r'Squared Distance $D^2$')
    plt.title('EKF Innovation Consistency Check (Mahalanobis Distance)')
    plt.grid(True, which='both', linestyle='--', alpha=0.5)
    plt.legend(loc='upper right')
    plt.tight_layout()
    plt.show()

except FileNotFoundError:
    print(f"Mahalanobis file not found at {mah_file}. Run the filter to generate it.")